<a href="https://colab.research.google.com/github/Sanath-cmd/Internship_ITT/blob/main/Algorithms/Fashion_MNIST_ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

In [4]:
torch.manual_seed= 42

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'The device in use {device}')

The device in use cuda


In [6]:
train_data = pd.read_csv('/content/drive/MyDrive/fashion-mnist_train.csv')
test_data = pd.read_csv('/content/drive/MyDrive/fashion-mnist_test.csv')

In [7]:
X_train, y_train = train_data.iloc[:, 1:].values, train_data.iloc[:, 0].values
X_test, y_test = test_data.iloc[:, 1:].values, test_data.iloc[:, 0].values

In [8]:
X_train = X_train/255
X_test = X_test/255

In [9]:
class CustomDataset(Dataset):
  def __init__(self, features, labels):
    self.features = torch.tensor(features, dtype= torch.float32)
    self.labels = torch.tensor(labels, dtype= torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self, idx):
    return self.features[idx], self.labels[idx]

In [10]:
train_dataset = CustomDataset(X_train, y_train)
test_dataset = CustomDataset(X_test, y_test)


In [11]:
print(y_train[:20])

[2 9 6 0 3 4 4 5 4 8 0 8 9 0 2 2 9 3 3 3]


In [12]:
train_loader = DataLoader(train_dataset, batch_size= 128, shuffle = True, pin_memory= True)
test_loader = DataLoader(test_dataset, batch_size= 128, shuffle = False, pin_memory= True)

In [21]:
class Model(nn.Module):

  def __init__(self, input_size= 784, h1 = 256, h2 = 64, output_size= 10):
    super().__init__()
    self.fc1 = nn.Linear(input_size, h1)
    self.bn1 = nn.BatchNorm1d(h1)

    self.fc2 = nn.Linear(h1, h2)
    self.bn2 = nn.BatchNorm1d(h2)
    self.fc3 = nn.Linear(h2, output_size)

    self.dropout = nn.Dropout(0.3)


  def forward(self, x):
    x = self.fc1(x)
    x = self.bn1(x)
    x = F.relu(x)
    x = self.dropout(x)

    x = self.fc2(x)
    x = self.bn2(x)
    x = F.relu(x)

    x = self.dropout(x)
    x = self.fc3(x)
    return x

In [22]:
epochs = 100
learning_rate = 0.01

In [23]:
model = Model()
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr= learning_rate, weight_decay= 1e-4)

In [24]:
for epoch in range(epochs):
  train_losses = []
  total_epoch_loss = 0
  for batch_features, batch_labels in train_loader:
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
    output = model(batch_features)
    loss = criterion(output, batch_labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total_epoch_loss = total_epoch_loss + loss.item()
    train_losses.append(total_epoch_loss/len(train_loader))

  avg_loss = total_epoch_loss/len(train_loader)
  print(f'Epoch: {epoch+1}, Loss: {avg_loss}')

Epoch: 1, Loss: 1.0060419352578203
Epoch: 2, Loss: 0.6259494327914232
Epoch: 3, Loss: 0.5392875578611899
Epoch: 4, Loss: 0.4984925560224285
Epoch: 5, Loss: 0.47370115723182904
Epoch: 6, Loss: 0.45430319884946857
Epoch: 7, Loss: 0.43517170201486616
Epoch: 8, Loss: 0.42148217946481603
Epoch: 9, Loss: 0.41127287216786385
Epoch: 10, Loss: 0.40163360138945997
Epoch: 11, Loss: 0.38981842962917745
Epoch: 12, Loss: 0.38244345132857244
Epoch: 13, Loss: 0.37488964196842617
Epoch: 14, Loss: 0.3719514451110795
Epoch: 15, Loss: 0.3633710600928203
Epoch: 16, Loss: 0.356387251221549
Epoch: 17, Loss: 0.3513265083402967
Epoch: 18, Loss: 0.3472218798485392
Epoch: 19, Loss: 0.34097322116274315
Epoch: 20, Loss: 0.3359014830673173
Epoch: 21, Loss: 0.332043537071773
Epoch: 22, Loss: 0.329934843917137
Epoch: 23, Loss: 0.32445389833023297
Epoch: 24, Loss: 0.32028490743403243
Epoch: 25, Loss: 0.31740569143788394
Epoch: 26, Loss: 0.3134484327932411
Epoch: 27, Loss: 0.3109375567578558
Epoch: 28, Loss: 0.30633812

In [25]:
model.eval()
total = 0
correct = 0
with torch.no_grad():
  for batch_features, batch_labels in test_loader:
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
    output = model(batch_features)
    _ , predicted = torch.max(output.data, 1)
    total += batch_labels.shape[0]
    correct += (predicted == batch_labels).sum().item()
print(correct/total)


0.9001
